In [11]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer

os.chdir('/store/carroll/sbgplants/')

In [23]:
# file paths
raw = 'data/raw'
doi = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018

out_folder = 'data/out_csv'

table = 'insitu_plot_event'

In [24]:
# load, prepare relevant raw tables - plot features
sample_site = pd.read_csv(os.path.join(doi, 'sample_site.csv'))

# fix mistake in raw data - lat, lon column names switched. To be fixed in ESS-DIVE
sample_site = sample_site.rename(columns={
    'Longitude': 'latitude',
    'Latitude': 'longitude'
})

# format collection date
sample_site['collection_date'] = pd.to_datetime(sample_site[['Year', 'Month', 'Day']])

# map gps_accuracy_approx
gps_acc = {
    'RTK': 0.01,
    'TrimbleGeoXT': 0.5
}
sample_site['GPS_source'] = sample_site['GPS_source'].map(gps_acc)

# map fc method
fc_method = {
    'Meadow': 'Quadrat',
    'Tree': 'Visual', ## update this to match an existing ENUM VALUE or add a new ENUM VALUE - ask Dana
    'Shrub': 'Visual'
}
sample_site['fractional_cover_method'] = sample_site['VegetationType'].map(fc_method)

# map floristic survey
floristic_survey = {
    'Meadow': 1,
    'Tree': 0,
    'Shrub': 0
}
sample_site['floristic_survey'] = sample_site['VegetationType'].map(floristic_survey)

# reproject plot center coordinates to crs used for flightlines
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32613', always_xy=True)
sample_site['longitude'], sample_site['latitude'] = transformer.transform(sample_site['longitude'].values, sample_site['latitude'].values)

sample_site

,SamplingArea,Campaign,SampleSiteCode,Month,Day,Year,latitude,longitude,EPSG,GPS_source,...,VegetationType,FieldVegHeightMax_cm,FieldVegHeightMedian_cm,SoilMoisture_%_1,SoilMoisture_%_2,SoilMoisture_%_3,Foliar_IGSN,collection_date,fractional_cover_method,floristic_survey
0,RM,ER18,001-ER18,6,14,2018,4.313906e+06,327909.504290,4326.0,0.01,...,Meadow,21.0,13.0,6.0,6.0,7.0,IER18005A,2018-06-14,Quadrat,1
1,RM,ER18,002-ER18,6,14,2018,4.313913e+06,327909.473751,4326.0,0.01,...,Meadow,42.0,26.0,4.0,6.0,4.0,IER18005B,2018-06-14,Quadrat,1
2,RM,ER18,003-ER18,6,14,2018,4.313922e+06,327904.468056,4326.0,0.01,...,Meadow,31.0,17.0,6.0,7.0,5.0,IER18005C,2018-06-14,Quadrat,1
3,RM,ER18,004-ER18,6,14,2018,4.313934e+06,327909.497829,4326.0,0.01,...,Meadow,55.0,32.0,5.0,7.0,8.0,IER18005D,2018-06-14,Quadrat,1
4,RM,ER18,005-ER18,6,14,2018,4.313932e+06,327898.540633,4326.0,0.01,...,Meadow,59.0,14.0,7.0,13.0,12.0,IER18005E,2018-06-14,Quadrat,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,GS,ER18,474-ER18,7,30,2018,4.304137e+06,322828.406631,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180056,2018-07-30,Visual,0
473,GS,ER18,475-ER18,7,30,2018,4.304106e+06,322865.138959,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180057,2018-07-30,Visual,0
474,GS,ER18,476-ER18,7,30,2018,4.304090e+06,322866.596029,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180058,2018-07-30,Visual,0
475,GS,ER18,477-ER18,7,30,2018,4.303914e+06,322893.906595,4326.0,0.50,...,Tree,NaN,NaN,NaN,NaN,NaN,IER180049,2018-07-30,Visual,0


In [25]:
# prepare & populate out table
out_table = pd.DataFrame(index = range(len(sample_site)))

out_table['insitu_plot_event_id'] = range(len(out_table))
out_table['plot_name'] = sample_site['SampleSiteCode']
out_table['plot_veg_type'] = sample_site['VegetationType']
out_table['team'] = None
out_table['collection_date'] = sample_site['collection_date']
out_table['latitude'] = sample_site['latitude']
out_table['longitude'] = sample_site['longitude']
out_table['gps_plot_orientation'] = 'center'
out_table['gps_horizontal_accuracy'] = sample_site['GPS_source']
out_table['fractional_cover_method'] = sample_site['fractional_cover_method']
out_table['plot_cover_photos'] = True
out_table['floristic_survey'] = sample_site['floristic_survey']
out_table['notes'] = None
out_table['campaign_name'] = 'East River 2018'

out_table

,insitu_plot_event_id,plot_name,plot_veg_type,team,collection_date,latitude,longitude,gps_plot_orientation,gps_horizontal_accuracy,fractional_cover_method,plot_cover_photos,floristic_survey,notes,campaign_name
0,0,001-ER18,Meadow,None,2018-06-14,4.313906e+06,327909.504290,center,0.01,Quadrat,True,1,None,East River 2018
1,1,002-ER18,Meadow,None,2018-06-14,4.313913e+06,327909.473751,center,0.01,Quadrat,True,1,None,East River 2018
2,2,003-ER18,Meadow,None,2018-06-14,4.313922e+06,327904.468056,center,0.01,Quadrat,True,1,None,East River 2018
3,3,004-ER18,Meadow,None,2018-06-14,4.313934e+06,327909.497829,center,0.01,Quadrat,True,1,None,East River 2018
4,4,005-ER18,Meadow,None,2018-06-14,4.313932e+06,327898.540633,center,0.01,Quadrat,True,1,None,East River 2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,472,474-ER18,Tree,None,2018-07-30,4.304137e+06,322828.406631,center,0.50,Visual,True,0,None,East River 2018
473,473,475-ER18,Tree,None,2018-07-30,4.304106e+06,322865.138959,center,0.50,Visual,True,0,None,East River 2018
474,474,476-ER18,Tree,None,2018-07-30,4.304090e+06,322866.596029,center,0.50,Visual,True,0,None,East River 2018
475,475,477-ER18,Tree,None,2018-07-30,4.303914e+06,322893.906595,center,0.50,Visual,True,0,None,East River 2018


In [26]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)